# Cleaning Dataset Hortikultura — Kabupaten Kediri

Notebook ini membersihkan 6 file hortikultura secara **spesifik per file** (bukan generik), karena tiap file
punya struktur yang beda. Ditulis berdasarkan isi asli tiap file, bukan tebakan.

**File yang dibersihkan:**
| File | Struktur asli |
|---|---|
| `Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv` | Wide: 1 baris = 1 kecamatan, kolom = tiap komoditas. Ada baris total "Kediri" |
| `Produksi_Hortikultura_Buah_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Produksi_Hortikultura_Sayur_per_kecamatan.csv` | Sama (wide, ada baris "Kediri") |
| `Luas_Panen_Hortikultura_Sayur.csv` | Long, tingkat **Kabupaten** saja (kolom: Jenis Tanaman, nilai) |
| `Total_Produksi_Hortikultura_Buah.csv` | Sama (long, tingkat Kabupaten) |
| `Total_Produksi_Hortikultura_Sayur.csv` | Sama (long, tingkat Kabupaten) |

File biofarmaka tidak diproses di notebook ini.

**Temuan dari pengecekan isi file:**
1. **Datanya bukan estimasi.** Baris "Kediri" di tiap file wide adalah total resmi, dan **sama persis** dengan
   jumlah 26 kecamatan. Ini dipakai sebagai validasi otomatis.
2. **Satuan tidak seragam** di file luas panen sayur: mayoritas **hektar**, jamur dalam **m²**. Kolom `satuan`
   disimpan eksplisit, tidak dipaksa jadi satu satuan.
3. **Label satuan Jamur tiram salah di sumber** (teks "meter persegi", kode "Ha"). **Dikoreksi ke M2** karena
   file kabupaten menulis "(m2)" dengan nilai yang sama (2948) dan dua kolom jamur lain berkode M2. Koreksi
   dicetak dan diverifikasi otomatis (cell 7 dan 10), tidak dilakukan diam-diam.
4. **Penulisan satuan diseragamkan** (`ha`/`Ha` jadi `Ha`, `m2`/`M2` jadi `M2`, `kw` dan `kw/qui` jadi `Kw`)
   supaya file per kecamatan dan file kabupaten konsisten.
5. **Nenas di Ngancar sangat dominan** (±96% nenas kabupaten dan ±61% seluruh produksi buah). Sudah dicek: sesuai
   file sumber. Satuan produksi buah adalah **kuintal**, bukan ton (ton = kuintal / 10). Lihat cell 11.
6. **Nama kecamatan "Kayen Kidul"** (pakai spasi) disamakan ke "Kayenkidul".

**Catatan soal "-" (strip):** dalam data BPS, "-" biasanya berarti **tidak ada produksi/luas panen**, sehingga
di sini diperlakukan sebagai **0**. Beda dengan `...` yang diperlakukan sebagai data tidak tersedia (NaN).

**Output** (`data/clean/hortikultura/`):
- `luas_panen_hortikultura_sayur_per_kecamatan.csv` → `kecamatan, komoditas, luas_panen, satuan`
- `produksi_hortikultura_buah_per_kecamatan.csv` → `kecamatan, komoditas, produksi_kuintal`
- `produksi_hortikultura_sayur_per_kecamatan.csv` → `kecamatan, komoditas, produksi_kuintal`
- `luas_panen_hortikultura_sayur_kabupaten.csv` → `komoditas, luas_panen, satuan`
- `total_produksi_hortikultura_buah_kabupaten.csv` → `komoditas, produksi_kuintal, satuan`
- `total_produksi_hortikultura_sayur_kabupaten.csv` → `komoditas, produksi_kuintal, satuan`


In [12]:
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# ---- KONFIGURASI (ubah di sini) ----
TAHUN_DUMMY = [2020, 2021, 2022, 2023, 2024]   # tahun data simulasi; [] jika semua asli
TAHUN_JIKA_TIDAK_ADA_KOLOM = 2025              # hanya untuk file satu tahun yang tidak punya kolom Tahun

JUMLAH_KECAMATAN = 26
NAMA_TOTAL = "Kediri"                          # baris total kabupaten di file per kecamatan
MASTER_KECAMATAN = [
    "Mojo", "Semen", "Ngadiluwih", "Kras", "Ringinrejo", "Kandat", "Wates", "Ngancar",
    "Plosoklaten", "Gurah", "Puncu", "Kepung", "Kandangan", "Pare", "Badas", "Kunjang",
    "Plemahan", "Purwoasri", "Papar", "Pagu", "Kayenkidul", "Gampengrejo", "Ngasem",
    "Banyakan", "Grogol", "Tarokan",
]
KOREKSI_KECAMATAN = {"Kayen Kidul": "Kayenkidul"}
# Salah label di sumber (teks "meter persegi", kode "Ha"). Dibuktikan oleh file kabupaten yang menulis "(m2)".
KOREKSI_SATUAN_LUAS = {"jamur tiram": "M2"}
SATUAN_BAKU = {"ha": "Ha", "m2": "M2", "kw": "Kw", "kw/qui": "Kw"}

# ---- LOKASI PROJECT ----
search_dir, PROJECT_ROOT = os.getcwd(), None
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    if os.path.dirname(search_dir) == search_dir:
        break
    search_dir = os.path.dirname(search_dir)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Folder 'data' tidak ditemukan. Jalankan notebook dari dalam repository.")

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUT_DIR = os.path.join(DATA_DIR, "clean", "hortikultura")
os.makedirs(OUT_DIR, exist_ok=True)
FOLDER_HASIL = {"clean", "processed", "derived"}   # tidak ikut dicari sebagai sumber
print("Project root:", PROJECT_ROOT, "\nOutput      :", OUT_DIR)

Project root: d:\SDT\Project Analitika Data\AgriDash-0.1 
Output      : d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\hortikultura


In [13]:
POLA_KOLOM = re.compile(r"^(Luas Panen|Produksi)\s+(.*?)\s*\(([^()]*)\)\s*\(([^()]*)\)$")
POLA_ANGKA = re.compile(r"-?\d+(\.\d+)?")
POLA_TAHUN = re.compile(r"(?:19|20)\d{2}")


def rapikan_nama(teks):
    # rapikan spasi & seragamkan huruf awal kata ("Jamur tiram" -> "Jamur Tiram"); kata kapital (TW) dibiarkan
    kata = re.sub(r"\s+", " ", str(teks)).strip().split(" ")
    return " ".join(k[:1].upper() + k[1:] if k.islower() else k for k in kata)


def parse_nilai(seri, konteks):
    # '-' = tidak ada (konvensi BPS) -> 0. Kosong / '...' / teks lain -> kosong (NaN) dan dilaporkan.
    mentah = seri.astype(str).str.strip()
    nilai = pd.Series(np.nan, index=seri.index, dtype=float)
    nilai[mentah == "-"] = 0.0
    angka = mentah.str.fullmatch(POLA_ANGKA)
    nilai[angka] = mentah[angka].astype(float)
    tak_terbaca = (~angka) & (mentah != "-")
    if tak_terbaca.any():
        print(f"⚠️  {konteks}: {tak_terbaca.sum()} sel bukan angka, contoh: "
              f"{mentah[tak_terbaca].unique()[:5].tolist()} (dijadikan kosong)")
    if (nilai < 0).any():
        raise ValueError(f"{konteks}: ada nilai negatif")
    return nilai


def deteksi_layout(path):
    kol = [c.strip() for c in pd.read_csv(path, encoding="utf-8-sig", nrows=0).columns]
    if kol[:2] == ["Tahun", "Kecamatan"]:
        return "kec_multi"
    if kol[0] == "Kecamatan":
        return "kec_tunggal"
    if "Tahun" in kol:
        return "kab_long"
    if len(kol) > 2 and all(POLA_TAHUN.search(c) for c in kol[1:]):
        return "kab_wide_tahun"
    if len(kol) == 2:
        return "kab_tunggal"
    return "tidak_dikenal"


def cari_file(wajib_ada, dilarang=()):
    dilarang = tuple(dilarang) + ("produktivitas",)      # file hasil hitung bukan data sumber
    kandidat = []
    for root, dirs, files in os.walk(DATA_DIR):
        dirs[:] = [d for d in dirs if d.lower() not in FOLDER_HASIL]
        for f in files:
            n = f.lower()
            if n.endswith(".csv") and all(k in n for k in wajib_ada) and not any(k in n for k in dilarang):
                kandidat.append(os.path.join(root, f))
    if not kandidat:
        raise FileNotFoundError(f"Tidak ada file dengan kata kunci {wajib_ada} di {DATA_DIR}")
    info = [(p, deteksi_layout(p)) for p in kandidat]
    info = [x for x in info if x[1] != "tidak_dikenal"] or info
    # utamakan file multi-tahun, lalu yang paling baru diubah
    info.sort(key=lambda x: (x[1] in ("kec_tunggal", "kab_tunggal"), -os.path.getmtime(x[0])))
    if len(info) > 1:
        print(f"⚠️  {len(info)} kandidat untuk {wajib_ada}; dipakai: {os.path.basename(info[0][0])}")
        for p, lay in info[1:]:
            print(f"      diabaikan: {os.path.basename(p)} ({lay})")
    if info[0][1] == "tidak_dikenal":
        raise ValueError(f"Layout file tidak dikenali: {info[0][0]}")
    return info[0]


print("✓ Fungsi bantu siap.")

✓ Fungsi bantu siap.


In [14]:
def baca_kecamatan(path, layout, awalan, satuan_diizinkan):
    # File wide: [Tahun,] Kecamatan, lalu satu kolom per komoditas (nama kolom memuat satuan).
    nama_file = os.path.basename(path)
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    df = df[~df.apply(lambda r: all(str(x).strip() == "" for x in r), axis=1)].reset_index(drop=True)
    punya_tahun = layout == "kec_multi"
    awal = 2 if punya_tahun else 1
    if not punya_tahun:
        print(f"ℹ️  {nama_file}: tidak ada kolom Tahun, dianggap tahun {TAHUN_JIKA_TIDAK_ADA_KOLOM}.")

    info = {}
    for kolom in df.columns[awal:]:
        m = POLA_KOLOM.match(kolom.strip())
        if not m:
            raise ValueError(f"{nama_file}: nama kolom tidak dikenali polanya: {kolom!r}")
        aw, komoditas, teks, kode = m.groups()
        if aw != awalan:
            raise ValueError(f"{nama_file}: kolom {kolom!r} bukan '{awalan}'")
        komoditas = rapikan_nama(komoditas)
        satuan = SATUAN_BAKU.get(kode.strip().lower(), kode.strip())
        if awalan == "Luas Panen" and komoditas.lower() in KOREKSI_SATUAN_LUAS:
            baru = KOREKSI_SATUAN_LUAS[komoditas.lower()]
            if baru != satuan:
                print(f"  ℹ️  {nama_file}: satuan '{komoditas}' dikoreksi {satuan!r} -> {baru!r}")
                satuan = baru
        elif "meter" in teks.lower() and satuan != "M2":
            print(f"  ⚠️  Label satuan '{komoditas}' tidak konsisten (teks {teks!r}, kode {kode!r}); tidak dikoreksi.")
        if satuan not in satuan_diizinkan:
            raise ValueError(f"{nama_file}: satuan kolom {kolom!r} = {satuan!r}, diizinkan {sorted(satuan_diizinkan)}")
        info[kolom] = (komoditas, satuan)

    idv = ["Tahun", "Kecamatan"] if punya_tahun else ["Kecamatan"]
    p = df.melt(id_vars=idv, value_vars=list(info), var_name="kolom", value_name="mentah")
    out = pd.DataFrame({
        "tahun": p["Tahun"].str.strip().astype(int) if punya_tahun else TAHUN_JIKA_TIDAK_ADA_KOLOM,
        "kecamatan": p["Kecamatan"].map(lambda s: re.sub(r"\s+", " ", s).strip()).replace(KOREKSI_KECAMATAN),
        "komoditas": p["kolom"].map(lambda k: info[k][0]),
        "satuan": p["kolom"].map(lambda k: info[k][1]),
        "nilai": parse_nilai(p["mentah"], nama_file),
    })
    return out


def baca_kabupaten(path, layout):
    # File long tingkat kabupaten: label komoditas memuat nama dua bahasa + satuan, mis. "Bawang Daun/Scallion(ha)".
    nama_file = os.path.basename(path)
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    kol = list(df.columns)
    if layout == "kab_tunggal":
        p = pd.DataFrame({"label": df[kol[0]], "mentah": df[kol[1]], "tahun": TAHUN_JIKA_TIDAK_ADA_KOLOM})
        print(f"ℹ️  {nama_file}: tidak ada informasi tahun, dianggap tahun {TAHUN_JIKA_TIDAK_ADA_KOLOM}.")
    elif layout == "kab_long":
        kolom_nama = [c for c in kol if c != "Tahun"][0]
        p = pd.DataFrame({"label": df[kolom_nama], "mentah": df[kol[-1]],
                          "tahun": df["Tahun"].str.strip().astype(int)})
    elif layout == "kab_wide_tahun":
        p = df.melt(id_vars=[kol[0]], value_vars=kol[1:], var_name="kolom_tahun", value_name="mentah")
        p["tahun"] = p["kolom_tahun"].map(lambda c: int(POLA_TAHUN.search(c).group()))
        p = p.rename(columns={kol[0]: "label"})[["label", "mentah", "tahun"]]
    else:
        raise ValueError(f"{nama_file}: layout kabupaten tidak didukung: {layout}")

    p["label"] = p["label"].map(lambda s: re.sub(r"\s+", " ", str(s)).strip())
    p = p[p["label"] != ""].reset_index(drop=True)
    pecah = p["label"].str.extract(r"^(.*)\(([^()]+)\)$")
    tanpa_satuan = pecah[1].isna()
    if tanpa_satuan.any():
        print(f"⚠️  {nama_file}: {tanpa_satuan.sum()} label tanpa satuan di akhir, contoh: "
              f"{p.loc[tanpa_satuan, 'label'].head(3).tolist()}")
    out = pd.DataFrame({
        "tahun": p["tahun"],
        "komoditas": pecah[0].fillna(p["label"]).str.strip(),
        "satuan": pecah[1].str.strip().str.lower().map(lambda s: SATUAN_BAKU.get(s, s) if isinstance(s, str) else None),
        "nilai": parse_nilai(p["mentah"], nama_file),
    })
    return out


def kunci_nama(nama):
    # kunci pencocokan antar file: nama Indonesia (sebelum '/'), huruf kecil
    return str(nama).split("/")[0].strip().lower()


print("✓ Fungsi baca siap.")

✓ Fungsi baca siap.


In [15]:
DATASET_KEC = {
    "luas_buah":      dict(wajib=["luas", "panen", "buah", "kecamatan"], dilarang=[], awalan="Luas Panen",
                           satuan={"Ha", "M2"}, nilai="luas_panen", pakai_satuan=True,
                           out="luas_panen_buah_per_kecamatan.csv"),
    "luas_sayur":     dict(wajib=["luas", "panen", "sayur", "kecamatan"], dilarang=[], awalan="Luas Panen",
                           satuan={"Ha", "M2"}, nilai="luas_panen", pakai_satuan=True,
                           out="luas_panen_sayur_per_kecamatan.csv"),
    "produksi_buah":  dict(wajib=["produksi", "buah", "kecamatan"], dilarang=["total"], awalan="Produksi",
                           satuan={"Kw"}, nilai="produksi_kuintal", pakai_satuan=False,
                           out="produksi_buah_per_kecamatan.csv"),
    "produksi_sayur": dict(wajib=["produksi", "sayur", "kecamatan"], dilarang=["total"], awalan="Produksi",
                           satuan={"Kw"}, nilai="produksi_kuintal", pakai_satuan=False,
                           out="produksi_sayur_per_kecamatan.csv"),
}
DATASET_KAB = {
    "luas_buah":      dict(wajib=["luas", "panen", "buah"], dilarang=["kecamatan"], nilai="luas_panen",
                           out="luas_panen_buah_kabupaten.csv"),
    "luas_sayur":     dict(wajib=["luas", "panen", "sayur"], dilarang=["kecamatan"], nilai="luas_panen",
                           out="luas_panen_sayur_kabupaten.csv"),
    "produksi_buah":  dict(wajib=["total", "produksi", "buah"], dilarang=[], nilai="produksi_kuintal",
                           out="total_produksi_buah_kabupaten.csv"),
    "produksi_sayur": dict(wajib=["total", "produksi", "sayur"], dilarang=[], nilai="produksi_kuintal",
                           out="total_produksi_sayur_kabupaten.csv"),
}

HASIL_KEC, TOTAL_KEC, HASIL_KAB = {}, {}, {}

# ---------- Per kecamatan ----------
for kode, cfg in DATASET_KEC.items():
    print(f"\n=== {kode} (per kecamatan) ===")
    path, layout = cari_file(cfg["wajib"], cfg["dilarang"])
    print("File:", os.path.basename(path), f"[{layout}]")
    d = baca_kecamatan(path, layout, cfg["awalan"], cfg["satuan"])

    # a) 26 kecamatan + 1 baris total di tiap tahun, nama sesuai master, tanpa duplikat
    per_tahun = d.groupby("tahun")["kecamatan"].nunique()
    salah = per_tahun[per_tahun != JUMLAH_KECAMATAN + 1]
    assert salah.empty, f"{kode}: jumlah kecamatan (+total) per tahun tidak {JUMLAH_KECAMATAN + 1}:\n{salah}"
    asing = set(d["kecamatan"]) - set(MASTER_KECAMATAN) - {NAMA_TOTAL}
    assert not asing, f"{kode}: nama kecamatan tidak dikenali: {asing}"
    assert not d.duplicated(["tahun", "kecamatan", "komoditas"]).any(), f"{kode}: ada baris ganda"

    # b) komoditas & satuan konsisten antar tahun
    n_komoditas = d.groupby("tahun")["komoditas"].nunique()
    if n_komoditas.nunique() > 1:
        print(f"⚠️  Jumlah komoditas berubah antar tahun: {n_komoditas.to_dict()}")
    satuan_ganda = d.groupby("komoditas")["satuan"].nunique()
    if (satuan_ganda > 1).any():
        print(f"⚠️  Satuan berubah antar tahun untuk: {satuan_ganda[satuan_ganda > 1].index.tolist()}")

    # c) total 'Kediri' harus sama dengan jumlah 26 kecamatan
    total = d[d["kecamatan"] == NAMA_TOTAL].set_index(["tahun", "komoditas"])["nilai"]
    jumlah = d[d["kecamatan"] != NAMA_TOTAL].groupby(["tahun", "komoditas"])["nilai"].sum()
    beda = (total - jumlah).abs()
    beda = beda[beda > 0.01]
    if len(beda):
        print(f"⚠️  {len(beda)} total 'Kediri' tidak sama dengan jumlah kecamatan, contoh:")
        print(beda.head(8).to_string())
    else:
        print("✓ Total 'Kediri' = jumlah 26 kecamatan (semua tahun & komoditas).")

    if d["nilai"].isna().any():
        print(f"⚠️  {d['nilai'].isna().sum()} nilai kosong setelah pembersihan.")

    TOTAL_KEC[kode] = d[d["kecamatan"] == NAMA_TOTAL].copy()
    HASIL_KEC[kode] = d[d["kecamatan"] != NAMA_TOTAL].copy()
    print(f"✓ {HASIL_KEC[kode]['tahun'].nunique()} tahun × {JUMLAH_KECAMATAN} kecamatan × "
          f"{HASIL_KEC[kode]['komoditas'].nunique()} komoditas = {len(HASIL_KEC[kode])} baris")

# ---------- Tingkat kabupaten ----------
for kode, cfg in DATASET_KAB.items():
    print(f"\n=== {kode} (kabupaten) ===")
    path, layout = cari_file(cfg["wajib"], cfg["dilarang"])
    print("File:", os.path.basename(path), f"[{layout}]")
    d = baca_kabupaten(path, layout)
    assert not d.duplicated(["tahun", "komoditas"]).any(), f"{kode}: ada pasangan tahun-komoditas ganda"
    n_per_tahun = d.groupby("tahun")["komoditas"].nunique()
    if n_per_tahun.nunique() > 1:
        print(f"⚠️  Jumlah komoditas berubah antar tahun: {n_per_tahun.to_dict()}")
    satuan_ganda = d.groupby("komoditas")["satuan"].nunique()
    if (satuan_ganda > 1).any():
        print(f"⚠️  Satuan berubah antar tahun untuk: {satuan_ganda[satuan_ganda > 1].index.tolist()}")
    if d["nilai"].isna().any():
        print(f"⚠️  {d['nilai'].isna().sum()} nilai kosong setelah pembersihan.")
    HASIL_KAB[kode] = d
    print(f"✓ {d['tahun'].nunique()} tahun × {d['komoditas'].nunique()} komoditas = {len(d)} baris")


=== luas_buah (per kecamatan) ===
File: Luas_Panen_Hortikultura_Buah_per_kecamatan.csv [kec_multi]
✓ Total 'Kediri' = jumlah 26 kecamatan (semua tahun & komoditas).
✓ 6 tahun × 26 kecamatan × 27 komoditas = 4212 baris

=== luas_sayur (per kecamatan) ===
File: Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv [kec_multi]
  ℹ️  Luas_Panen_Hortikultura_Sayur_per_kecamatan.csv: satuan 'Jamur Tiram' dikoreksi 'Ha' -> 'M2'
✓ Total 'Kediri' = jumlah 26 kecamatan (semua tahun & komoditas).
✓ 6 tahun × 26 kecamatan × 26 komoditas = 4056 baris

=== produksi_buah (per kecamatan) ===
File: Produksi_Hortikultura_Buah_per_kecamatan.csv [kec_multi]
✓ Total 'Kediri' = jumlah 26 kecamatan (semua tahun & komoditas).
✓ 6 tahun × 26 kecamatan × 27 komoditas = 4212 baris

=== produksi_sayur (per kecamatan) ===
File: Produksi_Hortikultura_Sayur_per_kecamatan.csv [kec_multi]
✓ Total 'Kediri' = jumlah 26 kecamatan (semua tahun & komoditas).
✓ 6 tahun × 26 kecamatan × 26 komoditas = 4056 baris

=== luas_buah (k

In [16]:
for kode in DATASET_KEC:
    kec = HASIL_KEC[kode].assign(kunci=lambda x: x["komoditas"].map(kunci_nama))
    kab = HASIL_KAB[kode].assign(kunci=lambda x: x["komoditas"].map(kunci_nama))
    j_kec = kec.groupby(["tahun", "kunci"])["nilai"].sum()
    j_kab = kab.groupby(["tahun", "kunci"])["nilai"].sum()

    cocok = j_kec.index.intersection(j_kab.index)
    hanya_kec = j_kec.index.difference(j_kab.index)
    hanya_kab = j_kab.index.difference(j_kec.index)
    selisih = (j_kec[cocok] - j_kab[cocok]).abs()
    tidak_sama = selisih[selisih > 0.01 + 1e-6 * j_kab[cocok].abs()]

    print(f"\n=== {kode}: kecamatan vs kabupaten ===")
    print(f"Komoditas-tahun yang dibandingkan: {len(cocok)}")
    if len(tidak_sama):
        print(f"⚠️  {len(tidak_sama)} nilai berbeda, contoh:")
        print(pd.DataFrame({"jumlah_kecamatan": j_kec[tidak_sama.index],
                            "file_kabupaten": j_kab[tidak_sama.index]}).head(8).to_string())
    else:
        print("✓ Semua nilai yang bisa dibandingkan sama.")
    if len(hanya_kec):
        print(f"ℹ️  Hanya di file kecamatan ({len(hanya_kec)}): {sorted(set(k for _, k in hanya_kec))[:8]}")
    if len(hanya_kab):
        print(f"ℹ️  Hanya di file kabupaten ({len(hanya_kab)}): {sorted(set(k for _, k in hanya_kab))[:8]}")


=== luas_buah: kecamatan vs kabupaten ===
Komoditas-tahun yang dibandingkan: 162
✓ Semua nilai yang bisa dibandingkan sama.

=== luas_sayur: kecamatan vs kabupaten ===
Komoditas-tahun yang dibandingkan: 156
✓ Semua nilai yang bisa dibandingkan sama.

=== produksi_buah: kecamatan vs kabupaten ===
Komoditas-tahun yang dibandingkan: 162
✓ Semua nilai yang bisa dibandingkan sama.

=== produksi_sayur: kecamatan vs kabupaten ===
Komoditas-tahun yang dibandingkan: 156
✓ Semua nilai yang bisa dibandingkan sama.


In [17]:
def tambah_dummy(d):
    d = d.copy()
    d["is_dummy"] = d["tahun"].isin(TAHUN_DUMMY)    # penanda data simulasi, jangan dihapus
    return d

for kode, cfg in DATASET_KEC.items():
    d = tambah_dummy(HASIL_KEC[kode]).rename(columns={"nilai": cfg["nilai"]})
    kolom = ["tahun", "kecamatan", "komoditas", cfg["nilai"]] + (["satuan"] if cfg["pakai_satuan"] else []) + ["is_dummy"]
    d = d[kolom].sort_values(["tahun", "komoditas", "kecamatan"]).reset_index(drop=True)
    d.to_csv(os.path.join(OUT_DIR, cfg["out"]), index=False, encoding="utf-8-sig")
    print(f"✓ {cfg['out']:<42} {len(d):>5} baris")

for kode, cfg in DATASET_KAB.items():
    d = tambah_dummy(HASIL_KAB[kode]).rename(columns={"nilai": cfg["nilai"]})
    d = d[["tahun", "komoditas", cfg["nilai"], "satuan", "is_dummy"]]
    d = d.sort_values(["tahun", "komoditas"]).reset_index(drop=True)
    d.to_csv(os.path.join(OUT_DIR, cfg["out"]), index=False, encoding="utf-8-sig")
    print(f"✓ {cfg['out']:<42} {len(d):>5} baris")

print("\nSelesai. Hasil di:", OUT_DIR)
if TAHUN_DUMMY:
    print(f"ℹ️  Tahun {TAHUN_DUMMY} ditandai is_dummy=True di semua file hasil.")

✓ luas_panen_buah_per_kecamatan.csv           4212 baris
✓ luas_panen_sayur_per_kecamatan.csv          4056 baris
✓ produksi_buah_per_kecamatan.csv             4212 baris
✓ produksi_sayur_per_kecamatan.csv            4056 baris
✓ luas_panen_buah_kabupaten.csv                162 baris
✓ luas_panen_sayur_kabupaten.csv               156 baris
✓ total_produksi_buah_kabupaten.csv            162 baris
✓ total_produksi_sayur_kabupaten.csv           156 baris

Selesai. Hasil di: d:\SDT\Project Analitika Data\AgriDash-0.1\data\clean\hortikultura
ℹ️  Tahun [2020, 2021, 2022, 2023, 2024] ditandai is_dummy=True di semua file hasil.
